# 13. Hybrid Recommendation Engine <a id="13-hybrid" name="13-hybrid"></a>

In [ ]:
def hybrid_recommendations(title, user_id=None, n=5,
                           content_weight=0.65, collab_weight=0.35,
                           genre_filter=None, min_year=None,
                           max_runtime=None, min_rating=None):
    """
    Hybrid recommendation: content-based (TF-IDF) + collaborative (KNN).
    Optional user personalisation via favourite genre boost.
    """
    matched = fuzzy_match(title, title_to_idx)
    if not matched:
        print(f"Not found: {title}"); return None
    idx = title_to_idx[matched]
    if isinstance(idx, pd.Series): idx = idx.iloc[0]

    # Content scores
    content_sim = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    # Collab scores
    collab_sim = np.zeros(len(df))
    movie_id = df.iloc[idx]["id"]
    if movie_id in mid_to_idx:
        kidx = mid_to_idx[movie_id]
        dists, indices = knn_model.kneighbors(
            user_item_sparse.T[kidx], n_neighbors=min(50, len(mid_to_idx)))
        for dist, i in zip(dists.flatten(), indices.flatten()):
            nb_id = idx_to_mid.get(i)
            if nb_id:
                nb_rows = df[df["id"] == nb_id].index
                if len(nb_rows):
                    collab_sim[nb_rows[0]] = 1 - dist

    # Hybrid score
    hybrid = content_weight * content_sim + collab_weight * collab_sim

    # User genre boost
    if user_id:
        user_ratings = df_ratings[df_ratings["user_id"] == user_id]
        if len(user_ratings) > 3:
            top_movies = user_ratings.nlargest(10, "rating")["movie_id"]
            fav_genres = []
            for mid in top_movies:
                row = df[df["id"]==mid]
                if len(row): fav_genres.extend(str(row["genres"].values[0]).split(","))
            top_genres = [g.strip() for g, _ in Counter(fav_genres).most_common(3)]
            for g in top_genres:
                mask = df["genres"].str.contains(g, case=False, na=False)
                hybrid[mask] *= 1.08

    scored = df.copy()
    scored["_hybrid"] = hybrid
    scored = scored[scored.index != idx]

    if genre_filter: scored = scored[scored["genres"].str.contains(genre_filter, case=False, na=False)]
    if min_year:     scored = scored[scored["release_year"] >= min_year]
    if max_runtime:  scored = scored[scored["runtime"] <= max_runtime]
    if min_rating:   scored = scored[scored["vote_average"] >= min_rating]

    results = scored.nlargest(n, "_hybrid")[
        ["title","release_year","primary_genre","vote_average","runtime","director","_hybrid","poster_path"]
    ].reset_index(drop=True)
    results.index += 1
    return results, df.iloc[idx]

print(" Hybrid Recommendations Demo:")
test_hybrid = [
    ("Star Wars", "U0001"), ("Forrest Gump", "U0042"), ("Inception", "U0100"),
    ("The Godfather", "U0200"), ("Finding Nemo", "U0300"),
]
for movie, uid in test_hybrid:
    print(f"\n '{movie}' | User: {uid}")
    print("-"*55)
    res = hybrid_recommendations(movie, user_id=uid, n=5)
    if res:
        recs, inp = res
        print(recs[["title","primary_genre","vote_average","_hybrid"]].to_string())

 Hybrid Recommendations Demo:

 'Star Wars' | User: U0001
-------------------------------------------------------
                          title primary_genre  vote_average  _hybrid
1       The Empire Strikes Back     Adventure          8.39     0.33
2            Return of the Jedi     Adventure          7.91     0.28
3  Star Wars: The Force Awakens     Adventure          7.25     0.25
4           Clash of the Titans     Adventure          6.90     0.24
5      Star Wars: The Last Jedi     Adventure          6.75     0.21

 'Forrest Gump' | User: U0042
-------------------------------------------------------
                  title primary_genre  vote_average  _hybrid
1     Letters to Juliet        Comedy          7.00     0.32
2  New York, I Love You        Comedy          5.94     0.29
3  Sleepless in Seattle        Comedy          6.70     0.24
4              Fearless         Drama          7.51     0.23
5     Love & Basketball        Comedy          7.34     0.23

 'Inception' | Use